In [36]:
import sympy as sp

In [37]:
def nol_eksak(expr):
    return sp.cancel(sp.together(expr)) == 0

def laporan(nama, lulus, info=""):
    status = "LULUS" if lulus else "GAGAL"
    print(f"[{status}] {nama} {info}")

In [38]:
# Simbol dan fungsi metrik

r, a, M, Lam = sp.symbols("r a M Lambda", real=True)
c = sp.symbols("c", real=True)          # c = cos(theta)
th = sp.symbols("theta", real=True)
E, L = sp.symbols("E L", real=True)
p_t, p_r, p_th, p_ph = sp.symbols("p_t p_r p_theta p_phi", real=True)

s2 = 1 - c**2
Q = 1 + Lam * a**2 / 3
Delta_r = (r**2 + a**2) * (1 - Lam * r**2 / 3) - 2 * M * r
Delta_th = 1 + Lam * a**2 * c**2 / 3
Sigma = r**2 + a**2 * c**2
P = r**2 + a**2

In [39]:
# Metrik kovarian

g_tt = (a**2 * Delta_th * s2 - Delta_r) / (Q**2 * Sigma)
g_phph = s2 / (Q**2 * Sigma) * (Delta_th * P**2 - a**2 * Delta_r * s2)
g_tph = - a * s2 / (Q**2 * Sigma) * (Delta_th * P - Delta_r)
g_rr = Sigma / Delta_r
g_thth = Sigma / Delta_th

g = sp.Matrix([
    [g_tt,  0,    0,      g_tph ],
    [0,     g_rr, 0,      0     ],
    [0,     0,    g_thth, 0     ],
    [g_tph, 0,    0,      g_phph]
])

In [40]:
# Determinan

D_sp = g_tt * g_phph - g_tph**2
D_eksak = -Delta_r * Delta_th * s2 / Q**4
laporan("Determinan sympy (t,phi): D = -Delta_r Delta_th sin^2 / Q^4",
        nol_eksak(D_sp - D_eksak))

[LULUS] Determinan sympy (t,phi): D = -Delta_r Delta_th sin^2 / Q^4 


In [41]:
# Metrik invers

G_tt = - Q**2 * (Delta_th * P**2 - a**2 * Delta_r * s2) / (Sigma * Delta_r * Delta_th)
G_tph = - Q**2 * a * (Delta_th * P - Delta_r) / (Sigma * Delta_r * Delta_th)
G_phph = Q**2 * (Delta_r - a**2 * Delta_th * s2) / (Sigma * Delta_r * Delta_th * s2)
G_rr = Delta_r / Sigma
G_thth = Delta_th / Sigma

G = sp.Matrix([
    [G_tt,  0,    0,      G_tph],
    [0,     G_rr, 0,      0],
    [0,     0,    G_thth, 0],
    [G_tph, 0,    0,      G_phph],
])

identitas = g * G
semua_benar = all(
    nol_eksak(identitas[i, j] - (1 if i == j else 0))
    for i in range(4) for j in range(4)
)

laporan("Metrik invers: g_{mu alpha} g^{alpha nu} = delta", semua_benar)

[LULUS] Metrik invers: g_{mu alpha} g^{alpha nu} = delta 


In [42]:
laporan("g^tt = g_phph / D", nol_eksak(G_tt - g_phph / D_sp))
laporan("g^tphi = -g_tphi / D", nol_eksak(G_tph + g_tph / D_sp))
laporan("g^phiphi = g_tt / D", nol_eksak(G_phph - g_tt / D_sp))

[LULUS] g^tt = g_phph / D 
[LULUS] g^tphi = -g_tphi / D 
[LULUS] g^phiphi = g_tt / D 
